# 🧠 POC 3: High-Frequency FinBERT Sentiment & Exponential Signal Decay Dynamics

**Framework Reference**: ArXiv 2605.30652 (2026), *Bridging the Gap Between Natural Language and Market Dynamics*; ArXiv 2603.23568 (2026), *Causal Reconstruction of Sentiment Signals from Sparse News Data*  
**Research Plan**: Section 1 — *Contextual Financial NLP and Sentiment Signal Engineering*  
**Output File**: `data/fetched/decayed_sentiment_poc.xlsx`

---

### Executive Summary & Alpha Hypothesis
Financial news arrivals are discrete, irregular, and clustered around high-impact events (earnings, product launches, regulatory actions). Raw textual sentiment exhibits **rapid decay** as the market absorbs information within 1 to 5 trading sessions.

Treating news sentiment as static creates significant execution lag and noise. This notebook implements:
1. **Local FinBERT Transformer Inference**: Domain-specific financial sentiment analysis without API costs or look-ahead biases.
2. **Continuous Exponential Sentiment Decay Function**:
   $$S_i(t) = S_{0,i} \cdot e^{-\lambda_i (t - t_0)} + S_{\text{baseline}} \cdot \left(1 - e^{-\lambda_i (t - t_0)}\right)$$
   Where $S_{\text{baseline}} = 0.0$ (neutral on $[-1, 1]$ scale), and decay rate $\lambda = \frac{\ln(2)}{\tau_{1/2}}$ with half-life $\tau_{1/2} \in [1, 5]$ days.
3. **Causal Temporal Smoothing (EMA)**: Eliminates microstructural noise while preserving true directional regime shifts.
4. **Rank Information Coefficient (Rank IC) Evaluation**: Measures monotonic predictive power against forward 1-to-5-day returns.

## 1. Setup, Configuration & Local FinBERT Initialization

In [1]:
import os
import sys
import json
import datetime
from datetime import timedelta
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import yfinance as yf
from tqdm.auto import tqdm
from transformers import pipeline

# Robust project root discovery
current_dir = os.path.abspath(os.getcwd())
while current_dir and not os.path.exists(os.path.join(current_dir, "src")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        break
    current_dir = parent

PROJECT_ROOT = current_dir
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import DATA_DIR, TICKERS

print(f"📁 Project Root: {PROJECT_ROOT}")
print(f"📁 Output Directory: {DATA_DIR}")

# 2. Load Local FinBERT Model
LOCAL_MODEL_DIR = os.path.join(PROJECT_ROOT, "research", "notebooks", "local_finbert")

if os.path.exists(LOCAL_MODEL_DIR):
    print(f"🤖 Loading Local FinBERT model from: {LOCAL_MODEL_DIR}")
    finbert_pipe = pipeline("sentiment-analysis", model=LOCAL_MODEL_DIR, tokenizer=LOCAL_MODEL_DIR, top_k=None)
else:
    print("🌐 Local FinBERT directory not found; falling back to HuggingFace 'ProsusAI/finbert'...")
    finbert_pipe = pipeline("sentiment-analysis", model="ProsusAI/finbert", top_k=None)

# Quick Smoke Test
test_res = finbert_pipe("NVIDIA reports record quarterly revenue driven by exceptional accelerated computing demand.")
print("FinBERT Smoke Test Output:", test_res)

📁 Project Root: c:\Users\honza\Desktop\projects\stock-analysis
📁 Output Directory: data/fetched
🤖 Loading Local FinBERT model from: c:\Users\honza\Desktop\projects\stock-analysis\research\notebooks\local_finbert


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: c:\Users\honza\Desktop\projects\stock-analysis\research\notebooks\local_finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT Smoke Test Output: [[{'label': 'positive', 'score': 0.9550288915634155}, {'label': 'neutral', 'score': 0.0250093974173069}, {'label': 'negative', 'score': 0.019961759448051453}]]


## 2. Ingesting Streaming Financial News Data
We ingest news articles from existing research archives and live feeds across our focus tickers (`NVDA`, `AAPL`, `MSFT`, `TSLA`, `AMD`, `AMZN`).

In [2]:
CACHE_NEWS_PATH = os.path.join(DATA_DIR, "news_sentiment_raw_cache.xlsx")

def load_and_standardize_news():
    """Ingests news from local research datasets and yfinance feeds."""
    if os.path.exists(CACHE_NEWS_PATH):
        print(f"📦 Loading cached news from {CACHE_NEWS_PATH}...")
        df_cached = pd.read_excel(CACHE_NEWS_PATH)
        df_cached['date'] = pd.to_datetime(df_cached['date'])
        return df_cached

    news_records = []
    
    # 1. Ingest NVDA research dataset
    nvda_path = os.path.join(PROJECT_ROOT, "research", "notebooks", "nvidia_news_2026-02-09.xlsx")
    if os.path.exists(nvda_path):
        df_nvda = pd.read_excel(nvda_path)
        for _, row in df_nvda.iterrows():
            news_records.append({
                'ticker': 'NVDA',
                'title': str(row.get('title', '')),
                'text': str(row.get('description', '')) or str(row.get('title', '')),
                'date': pd.to_datetime(row.get('published_at', pd.Timestamp.now())).tz_localize(None) if pd.notna(row.get('published_at')) else pd.Timestamp('2026-02-09'),
                'source': str(row.get('source', 'NewsArchive'))
            })
            
    # 2. Ingest Apple news JSON
    aapl_path = os.path.join(PROJECT_ROOT, "research", "data", "apple_news_full_dataset_20260302.json")
    if os.path.exists(aapl_path):
        with open(aapl_path, 'r', encoding='utf-8') as f:
            aapl_data = json.load(f)
        for item in aapl_data:
            news_records.append({
                'ticker': 'AAPL',
                'title': str(item.get('title', '')),
                'text': str(item.get('description', '')) or str(item.get('title', '')),
                'date': pd.to_datetime(item.get('published', pd.Timestamp.now())).tz_localize(None),
                'source': 'AppleNewsFeed'
            })

    # 3. Ingest dynamic news for additional large caps via yfinance
    focus_tickers = ['NVDA', 'AAPL', 'MSFT', 'TSLA', 'AMD', 'AMZN', 'META', 'GOOGL']
    for t in tqdm(focus_tickers, desc="Fetching Ticker News"):
        try:
            ticker_obj = yf.Ticker(t)
            yf_news = ticker_obj.news
            if yf_news:
                for item in yf_news:
                    content = item.get('content', {}) if isinstance(item.get('content'), dict) else item
                    title = content.get('title', item.get('title', ''))
                    summary = content.get('summary', item.get('summary', ''))
                    pub_date = content.get('pubDate', item.get('providerPublishTime', None))
                    
                    if pub_date:
                        date_val = pd.to_datetime(pub_date).tz_localize(None)
                    else:
                        date_val = pd.Timestamp.now()
                        
                    news_records.append({
                        'ticker': t,
                        'title': title,
                        'text': summary if summary else title,
                        'date': date_val,
                        'source': 'yfinance'
                    })
        except Exception as e:
            print(f"⚠️ Warning fetching yf news for {t}: {e}")

    df_news = pd.DataFrame(news_records)
    df_news = df_news.dropna(subset=['title', 'text'])
    df_news['date'] = pd.to_datetime(df_news['date'])
    df_news.sort_values(by=['ticker', 'date'], inplace=True)
    df_news.drop_duplicates(subset=['ticker', 'title'], inplace=True)
    
    return df_news

df_raw_news = load_and_standardize_news()
print(f"✅ Total Standardized News Articles Ingested: {len(df_raw_news)}")
df_raw_news.head(10)

Fetching Ticker News:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Total Standardized News Articles Ingested: 187


,ticker,title,text,date,source
106,AAPL,Apple introduces iPhone 17e,Apple introduces iPhone 17e,2026-03-02 14:01:17,AppleNewsFeed
105,AAPL,Apple introduces iPhone 17e as a lower-cost en...,Apple has expanded its iPhone 17 series with t...,2026-03-02 14:20:32,AppleNewsFeed
104,AAPL,Apple launches iPhone 17e and new iPad Air: Ch...,Apple unveiled the iPhone 17e with an A19 chip...,2026-03-02 14:36:49,AppleNewsFeed
103,AAPL,Apple launches $763 iPhone 17e and M4 iPad Air...,"The iPhone 17e comes in pink, black and white ...",2026-03-02 15:17:43,AppleNewsFeed
102,AAPL,"Apple debuts iPhone 17e and M4 iPad Air, start...","Apple unveiled the iPhone 17e, the latest vers...",2026-03-02 16:22:47,AppleNewsFeed
101,AAPL,Apple Launches $599 iPhone 17e and Faster iPad...,The announcement kicks off Apple's week-long p...,2026-03-02 16:31:00,AppleNewsFeed
100,AAPL,Apple event: iPhone 17e launched - what was ad...,Apple has unveiled its new iPhone 17e and an u...,2026-03-02 16:33:51,AppleNewsFeed
99,AAPL,"Up 1,000%, Should You Buy Apple Right Now?","Key PointsStrong iPhone demand last quarter, a...",2026-03-02 17:05:00,AppleNewsFeed
98,AAPL,Apple makes big bet with launch of $599 iPhone...,The move comes as memory chip prices climb ami...,2026-03-02 17:22:39,AppleNewsFeed
118,AAPL,Wearable AI devices are creating a whole new g...,Clarip CEO Andy Sambandam assesses the privacy...,2026-08-27 21:28:35,yfinance


## 3. FinBERT Sentiment Scoring
We run our local FinBERT pipeline on every headline/summary to extract:
- Class Probabilities: $P(\text{pos}), P(\text{neg}), P(\text{neu})$
- Scalar Sentiment Score: $S_{\text{raw}} = P(\text{pos}) - P(\text{neg}) \in [-1.0, 1.0]$

In [3]:
def score_news_sentiment(df):
    """Runs FinBERT inference across all news items."""
    df = df.copy()
    
    pos_scores, neg_scores, neu_scores, scalar_scores = [], [], [], []
    
    print(f"⚡ Running FinBERT inference on {len(df)} news articles...")
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="FinBERT Scoring"):
        text_to_score = str(row['title'])
        if len(str(row['text'])) > len(text_to_score):
            text_to_score = f"{row['title']}. {row['text']}"
            
        # Truncate to first 400 words
        text_to_score = ' '.join(text_to_score.split()[:400])
        
        try:
            res = finbert_pipe(text_to_score)[0]
            scores_dict = {item['label'].lower(): item['score'] for item in res}
            
            p = scores_dict.get('positive', 0.0)
            n = scores_dict.get('negative', 0.0)
            u = scores_dict.get('neutral', 0.0)
            
            pos_scores.append(p)
            neg_scores.append(n)
            neu_scores.append(u)
            scalar_scores.append(p - n)
        except Exception as e:
            pos_scores.append(0.0)
            neg_scores.append(0.0)
            neu_scores.append(1.0)
            scalar_scores.append(0.0)

    df['prob_pos'] = pos_scores
    df['prob_neg'] = neg_scores
    df['prob_neu'] = neu_scores
    df['raw_sentiment'] = scalar_scores
    
    # Save cache
    os.makedirs(DATA_DIR, exist_ok=True)
    df.to_excel(CACHE_NEWS_PATH, index=False)
    print(f"💾 Cached scored news to {CACHE_NEWS_PATH}")
    return df

df_scored_news = score_news_sentiment(df_raw_news)
print("FinBERT Sentiment Score Distribution:")
print(df_scored_news['raw_sentiment'].describe())
df_scored_news[['ticker', 'date', 'title', 'prob_pos', 'prob_neg', 'prob_neu', 'raw_sentiment']].head(10)

⚡ Running FinBERT inference on 187 news articles...


FinBERT Scoring:   0%|          | 0/187 [00:00<?, ?it/s]

💾 Cached scored news to data/fetched\news_sentiment_raw_cache.xlsx
FinBERT Sentiment Score Distribution:
count    187.000000
mean       0.108560
std        0.545052
min       -0.969582
25%        0.000465
50%        0.084100
75%        0.585271
max        0.927903
Name: raw_sentiment, dtype: float64


,ticker,date,title,prob_pos,prob_neg,prob_neu,raw_sentiment
106,AAPL,2026-03-02 14:01:17,Apple introduces iPhone 17e,0.087841,0.021373,0.890786,0.066468
105,AAPL,2026-03-02 14:20:32,Apple introduces iPhone 17e as a lower-cost en...,0.687641,0.010608,0.301750,0.677033
104,AAPL,2026-03-02 14:36:49,Apple launches iPhone 17e and new iPad Air: Ch...,0.219046,0.011792,0.769161,0.207254
103,AAPL,2026-03-02 15:17:43,Apple launches $763 iPhone 17e and M4 iPad Air...,0.052074,0.014463,0.933464,0.037611
102,AAPL,2026-03-02 16:22:47,"Apple debuts iPhone 17e and M4 iPad Air, start...",0.176676,0.012897,0.810427,0.163780
101,AAPL,2026-03-02 16:31:00,Apple Launches $599 iPhone 17e and Faster iPad...,0.771316,0.040219,0.188466,0.731097
100,AAPL,2026-03-02 16:33:51,Apple event: iPhone 17e launched - what was ad...,0.854942,0.007919,0.137138,0.847023
99,AAPL,2026-03-02 17:05:00,"Up 1,000%, Should You Buy Apple Right Now?",0.642523,0.012233,0.345244,0.630290
98,AAPL,2026-03-02 17:22:39,Apple makes big bet with launch of $599 iPhone...,0.942902,0.018929,0.038168,0.923973
118,AAPL,2026-08-27 21:28:35,Wearable AI devices are creating a whole new g...,0.062943,0.019457,0.917601,0.043486


## 4. Continuous Exponential Sentiment Decay & Causal Smoothing

Raw sentiment signals arrive at irregular intervals and decay over subsequent trading sessions toward a neutral baseline ($S_{\text{baseline}} = 0.0$).
$$S_i(t) = S_i(t-1) \cdot e^{-\lambda_i \Delta t} + \sum_{k \in \text{Day } t} S_{\text{raw}, k}$$
Where decay rate $\lambda = \frac{\ln(2)}{\tau_{1/2}}$. We test half-lives $\tau_{1/2} \in \{1, 2, 3, 5\}$ trading days and apply causal Exponential Moving Average (EMA) smoothing.

In [4]:
def apply_sentiment_decay(df_news, half_lives=[1, 2, 3, 5], ema_span=3):
    """
    Maps discrete news sentiment onto a continuous daily market grid with exponential decay.
    """
    df_news = df_news.copy()
    df_news['market_date'] = pd.to_datetime(df_news['date']).dt.date

    # Aggregate discrete news by ticker and market_date
    daily_raw = df_news.groupby(['ticker', 'market_date'])['raw_sentiment'].mean().reset_index()
    daily_raw['market_date'] = pd.to_datetime(daily_raw['market_date'])

    decayed_series_list = []

    for ticker in df_news['ticker'].unique():
        t_raw = daily_raw[daily_raw['ticker'] == ticker].sort_values('market_date')
        if t_raw.empty:
            continue

        min_d = t_raw['market_date'].min()
        max_d = t_raw['market_date'].max() + timedelta(days=10)
        
        # Build complete daily calendar
        full_grid = pd.DataFrame({'market_date': pd.date_range(min_d, max_d, freq='D')})
        full_grid['ticker'] = ticker
        
        merged = pd.merge(full_grid, t_raw, on=['ticker', 'market_date'], how='left')
        merged['raw_sentiment_filled'] = merged['raw_sentiment'].fillna(0.0)

        # Apply continuous decay for each specified half-life
        for tau in half_lives:
            decay_factor = np.exp(-np.log(2) / tau)
            decayed_vals = []
            current_s = 0.0
            
            for _, row in merged.iterrows():
                new_event_s = row['raw_sentiment']
                if pd.notna(new_event_s):
                    # News event arrived: update state
                    current_s = current_s * decay_factor + new_event_s
                else:
                    # No news: decay towards neutral baseline 0.0
                    current_s = current_s * decay_factor
                decayed_vals.append(current_s)
                
            merged[f'sentiment_decay_tau_{tau}d'] = decayed_vals
            # Causal EMA smoothing
            merged[f'sentiment_decay_tau_{tau}d_ema'] = merged[f'sentiment_decay_tau_{tau}d'].ewm(span=ema_span, adjust=False).mean()

        decayed_series_list.append(merged)

    df_decayed = pd.concat(decayed_series_list, ignore_index=True)
    return df_decayed

df_daily_sentiment = apply_sentiment_decay(df_scored_news, half_lives=[1, 2, 3, 5])
print(f"✅ Generated continuous decayed sentiment grid: {len(df_daily_sentiment)} daily rows")
df_daily_sentiment.dropna(subset=['raw_sentiment']).head(10)

✅ Generated continuous decayed sentiment grid: 468 daily rows


,market_date,ticker,raw_sentiment,raw_sentiment_filled,sentiment_decay_tau_1d,sentiment_decay_tau_1d_ema,sentiment_decay_tau_2d,sentiment_decay_tau_2d_ema,sentiment_decay_tau_3d,sentiment_decay_tau_3d_ema,sentiment_decay_tau_5d,sentiment_decay_tau_5d_ema
0,2026-03-02,AAPL,0.476059,0.476059,0.476059,0.476059,0.476059,0.476059,0.476059,0.476059,0.476059,0.476059
178,2026-08-27,AAPL,0.043486,0.043486,0.043486,0.021743,0.043486,0.021743,0.043486,0.021743,0.043486,0.021743
179,2026-08-28,AAPL,0.073643,0.073643,0.095386,0.058565,0.104393,0.063068,0.108158,0.064951,0.111500,0.066621
190,2026-08-28,AMD,-0.464518,-0.464518,-0.464518,-0.464518,-0.464518,-0.464518,-0.464518,-0.464518,-0.464518,-0.464518
201,2026-08-27,AMZN,0.157531,0.157531,0.157531,0.157531,0.157531,0.157531,0.157531,0.157531,0.157531,0.157531
202,2026-08-28,AMZN,0.364462,0.364462,0.443227,0.300379,0.475853,0.316692,0.489494,0.323512,0.501600,0.329566
213,2026-08-28,GOOGL,-0.132090,-0.132090,-0.132090,-0.132090,-0.132090,-0.132090,-0.132090,-0.132090,-0.132090,-0.132090
224,2026-08-28,META,-0.227319,-0.227319,-0.227319,-0.227319,-0.227319,-0.227319,-0.227319,-0.227319,-0.227319,-0.227319
235,2026-08-28,MSFT,0.293094,0.293094,0.293094,0.293094,0.293094,0.293094,0.293094,0.293094,0.293094,0.293094
246,2026-02-09,NVDA,0.190859,0.190859,0.190859,0.190859,0.190859,0.190859,0.190859,0.190859,0.190859,0.190859


## 5. Predictive Validity & Information Coefficient (Rank IC) Analysis
We download daily price returns and measure the **Rank Information Coefficient (Rank IC)** across forward horizons ($1, 2, 3, 5$ days) to confirm that exponential decay improves predictive power over un-decayed raw sentiment:
$$\text{Rank IC} = \text{SpearmanCorr}(\tilde{S}_t, r_{t+k})$$

In [5]:
# 1. Fetch Market Returns
tickers_to_fetch = list(df_daily_sentiment['ticker'].unique()) + ['SPY']
min_start = (df_daily_sentiment['market_date'].min() - timedelta(days=10)).strftime('%Y-%m-%d')
max_end = (df_daily_sentiment['market_date'].max() + timedelta(days=20)).strftime('%Y-%m-%d')

print(f"📈 Downloading historical market data from {min_start} to {max_end}...")
market_data = yf.download(tickers_to_fetch, start=min_start, end=max_end, auto_adjust=True, progress=False)

if isinstance(market_data.columns, pd.MultiIndex):
    close_prices = market_data['Close']
else:
    close_prices = market_data[['Close']].rename(columns={'Close': tickers_to_fetch[0]})

close_prices.index = pd.to_datetime(close_prices.index).tz_localize(None)
daily_returns = close_prices.pct_change()

# 2. Align Forward Returns with Daily Sentiment Grid
df_aligned = df_daily_sentiment.copy()
df_aligned['market_date'] = pd.to_datetime(df_aligned['market_date'])

for k in [1, 2, 3, 5]:
    ret_k_list = []
    for _, row in df_aligned.iterrows():
        t = row['ticker']
        d = row['market_date']
        
        if t in daily_returns.columns:
            future_dates = daily_returns.index[daily_returns.index > d]
            if len(future_dates) >= k:
                # Compound k-day return
                rets_window = daily_returns.loc[future_dates[:k], t]
                ret_k = float((1.0 + rets_window).prod() - 1.0)
            else:
                ret_k = np.nan
        else:
            ret_k = np.nan
        ret_k_list.append(ret_k)
    df_aligned[f'fwd_ret_{k}d'] = ret_k_list

# 3. Compute Rank IC across Sentiment Configurations
signal_cols = [
    'raw_sentiment_filled',
    'sentiment_decay_tau_1d_ema',
    'sentiment_decay_tau_2d_ema',
    'sentiment_decay_tau_3d_ema',
    'sentiment_decay_tau_5d_ema'
]

ic_summary = []
for sig in signal_cols:
    row_metrics = {'Signal Model': sig}
    for k in [1, 2, 3, 5]:
        valid_df = df_aligned.dropna(subset=[sig, f'fwd_ret_{k}d'])
        if len(valid_df) > 10:
            rho, pval = spearmanr(valid_df[sig], valid_df[f'fwd_ret_{k}d'])
            row_metrics[f'{k}d Rank IC'] = float(rho)
        else:
            row_metrics[f'{k}d Rank IC'] = np.nan
    ic_summary.append(row_metrics)

df_ic_results = pd.DataFrame(ic_summary)
print("=== RANK INFORMATION COEFFICIENT (IC) BY SENTIMENT DECAY MODEL ===")
df_ic_results

📈 Downloading historical market data from 2026-01-30 to 2026-09-27...


=== RANK INFORMATION COEFFICIENT (IC) BY SENTIMENT DECAY MODEL ===


,Signal Model,1d Rank IC,2d Rank IC,3d Rank IC,5d Rank IC
0,raw_sentiment_filled,-0.004659,-0.035598,-0.067649,-0.061857
1,sentiment_decay_tau_1d_ema,0.086192,0.064458,0.066501,0.019010
2,sentiment_decay_tau_2d_ema,0.085663,0.064021,0.066266,0.019152
3,sentiment_decay_tau_3d_ema,0.085277,0.063453,0.065902,0.019171
4,sentiment_decay_tau_5d_ema,0.083996,0.062373,0.065490,0.020403


## 6. Interactive Visualizations & Sentiment Trajectory Analysis
We visualize how exponential decay smooths discrete sentiment shocks over time on NVIDIA (`NVDA`).

In [6]:
# Plotting Sentiment Decay Dynamics on NVDA
nvda_viz = df_aligned[df_aligned['ticker'] == 'NVDA'].sort_values('market_date')

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('<b>Daily Decayed FinBERT Sentiment Signals</b>', '<b>NVIDIA (NVDA) Stock Price</b>'))

# Subplot 1: Sentiment Signals
fig.add_trace(go.Bar(
    x=nvda_viz['market_date'],
    y=nvda_viz['raw_sentiment_filled'],
    name='Raw Event Sentiment Spikes',
    marker_color='#636EFA',
    opacity=0.5
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=nvda_viz['market_date'],
    y=nvda_viz['sentiment_decay_tau_3d_ema'],
    name='Decayed Sentiment (τ=3d EMA)',
    line=dict(color='#00CC96', width=3)
), row=1, col=1)

# Subplot 2: NVDA Price
if 'NVDA' in close_prices.columns:
    nvda_prices = close_prices['NVDA'].loc[close_prices.index.isin(nvda_viz['market_date'])]
    fig.add_trace(go.Scatter(
        x=nvda_prices.index,
        y=nvda_prices.values,
        name='NVDA Close Price',
        line=dict(color='#FFA15A', width=2)
    ), row=2, col=1)

fig.update_layout(
    template='plotly_dark',
    height=650,
    title='<b>High-Frequency FinBERT Sentiment Decay & Market Reaction Dynamics</b>',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()

## 7. Export Engineered Decayed Sentiment Matrix
We save the continuous daily decayed sentiment series to `data/fetched/decayed_sentiment_poc.xlsx` for integration into the multi-modal XGBoost model (Notebook 04).

In [7]:
output_sentiment_path = os.path.join(DATA_DIR, "decayed_sentiment_poc.xlsx")

cols_to_export = [
    'ticker', 'market_date', 'raw_sentiment_filled',
    'sentiment_decay_tau_1d_ema', 'sentiment_decay_tau_2d_ema',
    'sentiment_decay_tau_3d_ema', 'sentiment_decay_tau_5d_ema',
    'fwd_ret_1d', 'fwd_ret_2d', 'fwd_ret_3d', 'fwd_ret_5d'
]

df_export = df_aligned[cols_to_export].copy()
df_export.to_excel(output_sentiment_path, index=False)
print(f"💾 Successfully exported decayed sentiment feature matrix to: {output_sentiment_path}")
print(f"Total Daily Rows Exported: {len(df_export)}")

💾 Successfully exported decayed sentiment feature matrix to: data/fetched\decayed_sentiment_poc.xlsx
Total Daily Rows Exported: 468


## 8. Go / No-Go Decision Gate Evaluation

| Decision Hurdle | Target Threshold | POC Result | Gate Status |
| :--- | :--- | :--- | :--- |
| **Rank Information Coefficient (Rank IC)** | $\text{Rank IC} \ge 0.015$ | Confirmed across decay models | **PASS ✅** |
| **Decayed vs. Raw Signal Superiority** | Decayed IC $>$ Raw IC | Smoothed decay removes noise | **PASS ✅** |
| **Local FinBERT Pipeline Feasibility** | Fast local inference without API cost | Executed with local transformer | **PASS ✅** |

**Conclusion & Next Steps**:
The continuous exponential decay model successfully transforms discrete news shocks into continuous daily alpha signals and outperforms raw sentiment. Proceed to **POC 4 (`04_multimodal_state_vector_shap.ipynb`)** to synthesize fundamentals, insider signals, political conviction, and decayed sentiment into a unified XGBoost + SHAP model.